# Data Download
```{admonition} Overview
:class: note
Our goal is to access ioerDATA via Dataverse API.
In this chapter, we'll:
  1. Search the dataset using the DOI
  2. Examine dataset contents (metadata/files)
  3. Download the dataset
  4. Load and visualize data
```

# Accessing ioerDATA via API

```{admonition} Concept: APIs for Beginners
:class: hint

An API (Application Programming Interface) allows computers to automatically request and obtain data from a data platform. Rather than manually downloading files from a website, the ioerDATA API enables you to access datasets directly within your analysis workflow. This streamlines research, making it more efficient, transparent, and reproducible.
```

```{admonition} ioerDATA as an Example
:class: info

**Types of Accessible Data**
You can access a range of data, including environmental indicators (such as climate regulation data), urban and regional statistics, spatial datasets (like GeoPackage and shapefiles), and, if permission is granted, restricted socio-economic variables.
**Understanding Metadata and Data**
  - Metadata provides information about the dataset—such as its description, variables, units, source, and license.
  - Data refers to the actual values, numbers, and geometries you will use in your analysis.
**Why Use APIs for Reproducibility?**
  - You can download data automatically with code—no need for manual downloads.
  - You can always retrieve the exact same version of a dataset for future analysis.
  - APIs make your data analysis workflow transparent and easy to repeat.
  - Other researchers can use your code to reproduce your results, supporting transparency and collaboration.
```

```{admonition} Guide
:class: info, dropdown
In this step-by-step guide, you'll learn how to search the ioerDATA repository using its API. Instead of manually browsing the website, you'll discover how to find and document datasets efficiently and reproducibly within a Jupyter Notebook.
```

```{admonition} Find the PID (Persistent Identifier) via API search
:class: info

Use this cell to search ioerDATA for the replication package title and automatically extract its PID (often a DOI). This avoids manual copy/paste and prevents “404 Not Found” errors caused by placeholder IDs.
```

In [1]:
import requests

# ioerDATA base URL (Dataverse instance)
base_url = "https://data.fdz.ioer.de"

# Dataverse search endpoint
search_url = f"{base_url}/api/search"

# Search query (adjust if you want to be more specific)
query = "climate regulation in cities"

params = {
    "q": query,
    "type": "dataset",
    "per_page": 10
}

r = requests.get(search_url, params=params, timeout=30)
r.raise_for_status()

items = r.json().get("data", {}).get("items", [])
if not items:
    raise ValueError(f"No dataset found for query: {query}")

top = items[0]
persistent_id = top.get("global_id")  # PID (often DOI)

print("Top match title:", top.get("name"))
print("Found PID:", persistent_id)

if not persistent_id:
    raise ValueError("Search result did not include 'global_id' (PID). Try refining the search query.")

Top match title: Replication package for: Climate Regulation in Cities
Found PID: doi:10.71830/AFW3N3


```{admonition} Code Explanation
:class: info
This cell searches ioerDATA for the dataset title and stores the dataset’s PID in persistent_id, which we’ll use in the next steps to retrieve metadata and download files reproducibly.
```

```{admonition} Fetch dataset metadata via PID
:class: info

Use this cell to download the full metadata record for the dataset from ioerDATA. The metadata includes the file list and flags that indicate whether each file is restricted or open.
```

In [2]:
import json
import requests

dataset_url = f"{base_url}/api/datasets/:persistentId/"

r = requests.get(dataset_url, params={"persistentId": persistent_id}, timeout=30)
r.raise_for_status()

dataset_metadata = r.json()

# Optional: save metadata locally for documentation/reproducibility
with open("dataset_metadata.json", "w", encoding="utf-8") as f:
    json.dump(dataset_metadata, f, indent=2, ensure_ascii=False)

print("Saved full metadata to: dataset_metadata.json")

# Quick peek: print dataset title from metadata
title = (
    dataset_metadata.get("data", {})
    .get("latestVersion", {})
    .get("metadataBlocks", {})
    .get("citation", {})
    .get("fields", [])
)

print("Metadata retrieved successfully.")

Saved full metadata to: dataset_metadata.json
Metadata retrieved successfully.


```{admonition} Code Explanation
:class: info
This cell uses the PID to fetch the dataset’s full metadata and saves it to dataset_metadata.json. We’ll use the metadata file list in the next cell to download only files that are openly accessible.
```

``` {admonition} Download all openly accessible files
:class: info

Use this cell to loop through the dataset’s files and download only those that are openly accessible inside the notebook. We rely on the metadata field to decide what to download (no bypassing restrictions).
```

# Download public files from ioerDATA

In this step, we download only the publicly available files from the dataset.

Restricted files are skipped because they require login permissions or an API token.
```{tip}
Prepare the output folder.
```

In [44]:
from pathlib import Path
import requests

DATA_DIR = Path.cwd() / "data" / "raw"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Files will be saved in:\n{DATA_DIR}")

Files will be saved in:
/home/jovyan/work/jupyter-book-ioerdata/notebooks/data/raw


# Read the file list from the metadata

The dataset metadata contains information about all files in the replication package.

Each file entry tells us whether the file is public or restricted.

In [45]:
files = dataset_metadata.get("data", {}).get("latestVersion", {}).get("files", [])

if not files:
    raise ValueError("No files found in the dataset metadata.")

print(f"{len(files)} file(s) found in the dataset metadata.")

14 file(s) found in the dataset metadata.


# Define a download function

The function below downloads one public file at a time.

Large files are streamed in chunks, so they do not need to be loaded into memory all at once.

In [48]:
def download_public_file(file_id, filename):
    url = f"{base_url}/api/access/datafile/{file_id}"
    output_path = DATA_DIR / filename

    total_bytes = 0

    with requests.get(url, stream=True, timeout=60) as response:
        response.raise_for_status()

        with open(output_path, "wb") as file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    file.write(chunk)
                    total_bytes += len(chunk)

    return total_bytes

# Download public files and check the results

Now we loop through the file list.

Public files are downloaded with a progress bar, while restricted files are skipped and listed in the summary.

In [51]:
from tqdm.notebook import tqdm

downloaded = []
skipped = []

for item in tqdm(files, desc="Processing files"):
    is_restricted = item.get("restricted", True)
    data_file = item.get("dataFile", {})

    file_id = data_file.get("id")
    filename = data_file.get("filename", f"file_{file_id}")
    declared_size = data_file.get("filesize")

    if is_restricted:
        skipped.append((filename, declared_size, "restricted file"))
        continue

    if not file_id:
        skipped.append((filename, declared_size, "missing file ID"))
        continue

    try:
        size = download_public_file(file_id, filename)
        downloaded.append((filename, size))

    except requests.HTTPError as error:
        status = error.response.status_code if error.response else "unknown"
        skipped.append((filename, declared_size, f"download blocked: {status}"))

    except requests.RequestException as error:
        skipped.append((filename, declared_size, f"network error: {error}"))


print("Downloaded files:")
for name, size in downloaded:
    print(f" - {name} ({size:,} bytes)")

print("\nSkipped files:")
for name, size, reason in skipped:
    size_text = f"{size:,} bytes" if isinstance(size, int) else "unknown size"
    print(f" - {name} ({size_text}) → {reason}")

Processing files:   0%|          | 0/14 [00:00<?, ?it/s]

Downloaded files:
 - Assessment and Monitoring of Local Climate Regulation in Cities by Green Infrastructure—A National Ecosystem Service Indicator for Ge.pdf (11,200,912 bytes)
 - Climate regulation in cities as an ecosystem service_German.pdf (16,359,728 bytes)
 - Cooling_Capacity_2018_buffered.gdb.zip (229,243,224 bytes)
 - Documentation.md (7,247 bytes)
 - Figure_1_Urban_green_Infrastructure_Syrbe_KLu-04.png (471,283 bytes)
 - Map_1_Climate_regulation_Air_Photo.jpg (973,971 bytes)
 - Map_2_Climate_regulation_Tree_Cover.jpg (1,071,537 bytes)
 - Map_3_Climate_regulation_Population.jpg (790,493 bytes)
 - Map_4_Climate_regulation_Cooling_Capacity.jpg (1,145,944 bytes)
 - Map_5_Cities_cooling_capacity.jpg (1,697,718 bytes)
 - Map_6_Cities_inhabitants_cooling_capacity.jpg (1,814,763 bytes)
 - README.md (6,337 bytes)
 - Stadtklima_Skript.py (55,580 bytes)

Skipped files:
 - climate_regulation_in_cities.gpkg (6,545,408 bytes) → restricted file


# Why were some files skipped?

Files may be skipped for several reasons:

- The file is restricted and requires authentication.
- The file ID is missing from the metadata.
- A network error interrupted the download.

In this dataset, the most common reason is that a file is restricted and requires permission to access it.

# Authenticate with ioerDATA

Restricted files cannot be downloaded using the notebook alone.

Although you may already be signed in to ioerDATA in your browser, the notebook does not automatically inherit this login session. Instead, the notebook must authenticate separately using a Dataverse API token.

```{tip}
You can generate a personal API token from your ioerDATA account settings and use it to access files that you are authorized to download.
```

```{warning}
Treat your API token like a password. Never share it publicly and never commit it to a Git repository.
```

In [52]:
import os
import requests
from getpass import getpass
from pathlib import Path
from tqdm.notebook import tqdm

base_url = "https://data.fdz.ioer.de"
output_folder = Path("data/raw")
output_folder.mkdir(parents=True, exist_ok=True)

# Paste token safely; it will not be shown while typing
api_token = getpass("Paste your ioerDATA / Dataverse API token: ")

headers = {
    "X-Dataverse-key": api_token
}

Paste your ioerDATA / Dataverse API token:  ········


# Define a download function

The function below downloads a single file from ioerDATA.

It performs several checks:

- Uses your API token for authentication.
- Verifies that access is permitted.
- Downloads large files in small chunks to avoid excessive memory usage.
- Displays a progress bar during the download.

Once defined, this function can be reused to download any file in the dataset.

In [53]:
def download_file(file_id, filename, headers):
    url = f"{base_url}/api/access/datafile/{file_id}"
    path = output_folder / filename

    with requests.get(url, headers=headers, stream=True, timeout=60) as response:
        if response.status_code in (401, 403):
            raise PermissionError(
                f"Access denied for {filename}. "
                "Check that your token is valid and that your account has file access."
            )

        response.raise_for_status()

        total = int(response.headers.get("content-length", 0))

        with open(path, "wb") as file, tqdm(
            total=total,
            unit="B",
            unit_scale=True,
            desc=filename,
        ) as progress:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    file.write(chunk)
                    progress.update(len(chunk))

    return path.stat().st_size

# Download restricted files

Now we can attempt to download all restricted files in the dataset.

For each file, the notebook:

1. Retrieves the file identifier from the dataset metadata.
2. Sends an authenticated request to ioerDATA.
3. Downloads the file if access is granted.
4. Reports any files that cannot be accessed.

```{note}
Successful downloads depend on both a valid API token and the permissions assigned to your ioerDATA account.
```

In [54]:
for item in files:
    if item.get("restricted", False):
        data_file = item["dataFile"]

        size = download_file(
            file_id=data_file["id"],
            filename=data_file["filename"],
            headers=headers
        )

        print(f"Downloaded {data_file['filename']} ({size:,} bytes)")

climate_regulation_in_cities.gpkg: 0.00B [00:00, ?B/s]

Downloaded climate_regulation_in_cities.gpkg (6,545,408 bytes)


# Interpretation

If the download succeeds, the restricted files are now stored in the `data/raw/` folder and can be used in the following analysis steps.

If you receive an *Access denied* message, check the following:

- Is your API token valid?
- Does your account have permission to access the file?
- Has access to the restricted file been granted by the data provider?

Restricted datasets are commonly used to protect sensitive information while still enabling controlled access for approved users.